# Geo-VLA — train the vision models on a free GPU

Runtime → Change runtime type → **T4 GPU**. Then run the cells in order.

| Model | Dataset | Time on a T4 |
|---|---|---|
| ResNet-50 land-cover classifier | EuroSAT RGB (27k patches, downloaded automatically) | ~10 min / 10 epochs |
| Siamese U-Net change detector | LEVIR-CD (10k patches, upload once) | ~1 h / 50 epochs |

Every run saves full analytics (dataset statistics, architecture, training curves, confusion
matrix, per-class metrics, learned filters, feature maps, embeddings, prediction galleries).
Download them at the end for your slides, and upload the checkpoints in the Geo-VLA **Training studio**.

In [ ]:
!git clone -b claude/new-project-setup-x2fnaz https://github.com/koushik2456/Geo-vla.git
%cd Geo-vla
!pip install -q -r requirements.txt
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to GPU")

## 1 · Land-cover classifier on EuroSAT
Downloads EuroSAT automatically, trains, evaluates and registers the model.

In [ ]:
!python -m training.pipeline --real --model classifier --epochs 10 --batch-size 128

## 2 · Change detector on LEVIR-CD (optional)
Upload `LEVIR-CD.zip` (from https://justchenhao.github.io/LEVIR/ or Kaggle) to the Colab file panel, then:

In [ ]:
!python -m training.download_data levir --from /content/LEVIR-CD.zip
!python -m training.pipeline --real --model change_detector --epochs 50 --batch-size 16

## 3 · See the analytics here

In [ ]:
from IPython.display import Image, display
import glob, os
for d in sorted(glob.glob("models/registry/*/v*_analytics")):
    print(d)
    for name in ["training_curves.png", "confusion_matrix.png", "predictions.png", "filters.png", "feature_maps.png"]:
        p = os.path.join(d, name)
        if os.path.exists(p):
            display(Image(p, width=900))

## 4 · Download checkpoints + analytics
Upload each `.pth` (with its `.metrics.json`) in the Training studio → *Upload a checkpoint*.

In [ ]:
!cd models && zip -qr /content/geo_vla_models.zip registry checkpoints
from google.colab import files
files.download("/content/geo_vla_models.zip")